In [ ]:
# Installation des bibliothèques nécessaires
!pip install ultralytics roboflow


In [ ]:
from roboflow import Roboflow

# Connexion à votre projet Roboflow (gaetano v4)
rf = Roboflow(api_key="FclaP7kXAJ9yTNCxwyCy")
project = rf.workspace("gaetano").project("car-damage-segmentation-zzed4")
version = project.version(4)
dataset = version.download("yolov8")

print(f"\n✅ Terminé ! Le fichier configuration est ici : {dataset.location}/data.yaml")


In [ ]:
from ultralytics import YOLO
import torch

# Chargement du modèle de base (Nano Segmentation)
model = YOLO('yolov8n-seg.pt')

# Lancement de l'apprentissage sur vos images
# Cela va créer un dossier 'runs/segment/train/'
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    plots=True
)


In [ ]:
# --- ÉTAPE D'ÉVALUATION DÉTAILLÉE ---
# On lance la validation sur le jeu de test pour voir si le modèle est 'solide'
metrics = model.val()

print(f"\n--- STATISTIQUES DE PRÉCISION ---")
print(f"Précision (P): {metrics.seg.p:.4f} (Proportion de détections correctes)")
print(f"Rappel (Recall): {metrics.seg.r:.4f} (Proportion de dommages trouvés)")
print(f"mAP@0.5: {metrics.seg.map50:.4f} (Score global de précision)")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Visualisation des graphiques de performance
results_path = "runs/segment/train/results.png"
if os.path.exists(results_path):
    plt.figure(figsize=(20, 15))
    img = mpimg.imread(results_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title("ANALYSE DE L'ENTRAÎNEMENT (LOSS & mAP)", fontsize=20, fontweight='bold')
    plt.show()
else:
    print("Graphique results.png non trouvé. L'entraînement est-il bien fini ?")


In [ ]:
# Cette commande affiche le chemin vers votre fichier final
import os
print("--- ENTRAÎNEMENT TERMINÉ ---")
print("Votre modèle entraîné est ici :")
print(os.path.abspath("runs/segment/train/weights/best.pt"))
